Завдання 1. Довідник клієнтів у форматі Delta

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.customers_delta;

CREATE TABLE workspace.default.customers_delta AS
SELECT c_custkey, c_name, c_nationkey, c_mktsegment, c_acctbal 
FROM samples.tpch.customer
WHERE c_custkey BETWEEN 1 AND 20;

DESCRIBE DETAIL workspace.default.customers_delta;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,9ec86cbb-8cbd-4531-860c-d4832a0806b6,workspace.default.customers_delta,null,,2026-11-24T21:20:54.477Z,2026-11-24T21:20:56.000Z,List(),List(),1,2178,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


Завдання 2. Зміни в довіднику

In [0]:
%sql
INSERT INTO workspace.default.customers_delta 
SELECT c_custkey, c_name, c_nationkey, c_mktsegment, c_acctbal 
FROM samples.tpch.customer
WHERE c_custkey BETWEEN 21 AND 25;

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
UPDATE workspace.default.customers_delta
SET c_mktsegment = 'AUTOMOBILE'
WHERE c_custkey = 1;

num_affected_rows
1


In [0]:
%sql
DELETE FROM workspace.default.customers_delta
WHERE c_acctbal < 0;

num_affected_rows
1


In [0]:
%sql
SELECT * FROM workspace.default.customers_delta
ORDER BY c_custkey;

c_custkey,c_name,c_nationkey,c_mktsegment,c_acctbal
1,Customer#000000001,15,AUTOMOBILE,711.56
2,Customer#000000002,13,AUTOMOBILE,121.65
3,Customer#000000003,1,AUTOMOBILE,7498.12
4,Customer#000000004,4,MACHINERY,2866.83
5,Customer#000000005,3,HOUSEHOLD,794.47
6,Customer#000000006,20,AUTOMOBILE,7638.57
7,Customer#000000007,18,AUTOMOBILE,9561.95
8,Customer#000000008,17,BUILDING,6819.74
9,Customer#000000009,8,FURNITURE,8324.07
10,Customer#000000010,5,HOUSEHOLD,2753.54


In [0]:
%sql
DESCRIBE HISTORY workspace.default.customers_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-11-24T21:21:04.000Z,73609044245796,tyshykviktoria77@gmail.com,DELETE,"Map(predicate -> [""(c_acctbal#19378 < 0.00)""])",null,List(1713592598634126),d0842e9d-c716-4457-b6e8-856531c96fba,1124-204453-jwnvn1nd-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 954, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 535, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 417)",null,Databricks-Runtime/19.16.x-aarch64-photon-scala2.13
2,2026-11-24T21:21:02.000Z,73609044245796,tyshykviktoria77@gmail.com,UPDATE,"Map(predicate -> [""(c_custkey#18946L = 1)""])",null,List(1713592598634126),47128230-3901-4d3d-a162-773e5ab4d0a7,1124-204453-jwnvn1nd-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2000, numDeletionVectorsUpdated -> 0, scanTimeMs -> 692, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1859, rewriteTimeMs -> 1306)",null,Databricks-Runtime/19.16.x-aarch64-photon-scala2.13
1,2026-11-24T21:20:59.000Z,73609044245796,tyshykviktoria77@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1713592598634126),15cb790c-9837-4074-8c32-cf4797565c2a,1124-204453-jwnvn1nd-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1978)",null,Databricks-Runtime/19.16.x-aarch64-photon-scala2.13
0,2026-11-24T21:20:56.000Z,73609044245796,tyshykviktoria77@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1713592598634126),7cd3fb99-ccf1-471a-b862-b4c1ab39357d,1124-204453-jwnvn1nd-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 2178)",null,Databricks-Runtime/19.16.x-aarch64-photon-scala2.13


Завдання 3. Пакет змін із CRM

In [0]:
%sql
MERGE INTO workspace.default.customers_delta AS cd
USING (
    SELECT c_custkey, c_name, c_nationkey, c_mktsegment, c_acctbal+100 AS c_acctbal
    FROM samples.tpch.customer
    WHERE c_custkey BETWEEN 21 AND 30
) AS p
ON cd.c_custkey = p.c_custkey
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT * FROM workspace.default.customers_delta
ORDER BY c_custkey;

c_custkey,c_name,c_nationkey,c_mktsegment,c_acctbal
1,Customer#000000001,15,AUTOMOBILE,711.56
2,Customer#000000002,13,AUTOMOBILE,121.65
3,Customer#000000003,1,AUTOMOBILE,7498.12
4,Customer#000000004,4,MACHINERY,2866.83
5,Customer#000000005,3,HOUSEHOLD,794.47
6,Customer#000000006,20,AUTOMOBILE,7638.57
7,Customer#000000007,18,AUTOMOBILE,9561.95
8,Customer#000000008,17,BUILDING,6819.74
9,Customer#000000009,8,FURNITURE,8324.07
10,Customer#000000010,5,HOUSEHOLD,2753.54


Завдання 4. Історія версій

In [0]:
history_df = spark.sql("""
    DESCRIBE HISTORY workspace.default.customers_delta
""")

result_df = history_df.select(
    "version",
    "operation",
    "operationMetrics"
).orderBy("version")

display(result_df)

version,operation,operationMetrics
0,CREATE TABLE AS SELECT,"Map(numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 2178)"
1,WRITE,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1978)"
2,UPDATE,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2000, numDeletionVectorsUpdated -> 0, scanTimeMs -> 692, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1859, rewriteTimeMs -> 1306)"
3,DELETE,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 954, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 535, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 417)"
4,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2100, numTargetBytesRemoved -> 1978, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 2059, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 882, numTargetRowsUpdated -> 5, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1139)"


Завдання 5. Time travel

Завдання 5.1. Клієнти і баланс за версіями

In [0]:
%sql
SELECT 0 AS version, COUNT(*) AS customers_cnt, SUM(c_acctbal) AS total_acctbal 
FROM workspace.default.customers_delta VERSION AS OF 0 
UNION ALL
SELECT 1 AS version, COUNT(*) AS customers_cnt, SUM(c_acctbal) AS total_acctbal 
FROM workspace.default.customers_delta VERSION AS OF 1 
UNION ALL
SELECT 2 AS version, COUNT(*) AS customers_cnt, SUM(c_acctbal) AS total_acctbal 
FROM workspace.default.customers_delta VERSION AS OF 2
UNION ALL
SELECT 3 AS version, COUNT(*) AS customers_cnt, SUM(c_acctbal) AS total_acctbal 
FROM workspace.default.customers_delta VERSION AS OF 3
UNION ALL
SELECT 4 AS version, COUNT(*) AS customers_cnt, SUM(c_acctbal) AS total_acctbal 
FROM workspace.default.customers_delta VERSION AS OF 4
ORDER BY version;

version,customers_cnt,total_acctbal
0,20,88826.46
1,25,110568.08
2,25,110568.08
3,24,110840.68
4,29,140649.03


Завдання 5.2. Стан на момент видалення

In [0]:
history_df = spark.sql("""
    DESCRIBE HISTORY workspace.default.customers_delta
""")

delete_timestamp = (
    history_df
    .filter("operation = 'DELETE'")
    .select("timestamp")
    .first()["timestamp"]
)

as_of = delete_timestamp.strftime("%Y-%m-%d %H:%M:%S.%f")

result_df = spark.sql(f"""
    SELECT
        TIMESTAMP '{as_of}' AS as_of,
        COUNT(*) AS customers_cnt
    FROM workspace.default.customers_delta
    TIMESTAMP AS OF '{as_of}'
""")

display(result_df)


as_of,customers_cnt
2026-11-24T21:21:04.000Z,24


Завдання 6. Помилка і відновлення

Завдання 6.1. Повернення до попередньої версії

In [0]:
%sql
UPDATE workspace.default.customers_delta
SET c_acctbal = 0;

num_affected_rows
29


In [0]:
%sql
RESTORE TABLE workspace.default.customers_delta TO VERSION AS OF 4;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
6137,3,1,3,2191,6137


Завдання 6.2. Перевірка повернення

In [0]:
%sql
SELECT 
    ( 
        SELECT COUNT(*) FROM (
            SELECT * FROM workspace.default.customers_delta 
            EXCEPT
            SELECT * FROM workspace.default.customers_delta VERSION AS OF 4
        )
    ) AS only_current,
    (
        SELECT COUNT(*) FROM (
            SELECT * FROM workspace.default.customers_delta VERSION AS OF 4
            EXCEPT
            SELECT * FROM workspace.default.customers_delta
        )
    ) AS only_merge_version;

only_current,only_merge_version
0,0


Завдання 7. Обслуговування таблиці

In [0]:
%sql
DESCRIBE DETAIL workspace.default.customers_delta;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,9ec86cbb-8cbd-4531-860c-d4832a0806b6,workspace.default.customers_delta,null,,2026-11-24T21:20:54.477Z,2026-11-24T21:21:30.000Z,List(),List(),3,6137,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 2, numDeletionVectors -> 1)",false


In [0]:
%sql
OPTIMIZE workspace.default.customers_delta;

path,metrics
,"List(1, 3, List(2255, 2255, 2255.0, 1, 2255), List(1859, 2178, 2045.6666666666667, 3, 6137), 0, null, null, 0, 1, 3, 0, true, 0, 0, 1795555296463, 1795555298943, 8, 1, null, List(1, 2), null, 5, 5, 294, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE DETAIL workspace.default.customers_delta;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,9ec86cbb-8cbd-4531-860c-d4832a0806b6,workspace.default.customers_delta,null,,2026-11-24T21:20:54.477Z,2026-11-24T21:21:39.000Z,List(),List(),1,2255,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
VACUUM workspace.default.customers_delta;

path
""


In [0]:
history_df = spark.sql("""
    DESCRIBE HISTORY workspace.default.customers_delta
""")

result_df = history_df.select(
    "version",
    "operation",
    "operationMetrics"
).orderBy("version")

display(result_df)

version,operation,operationMetrics
0,CREATE TABLE AS SELECT,"Map(numFiles -> 1, numOutputRows -> 20, numOutputBytes -> 2178)"
1,WRITE,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1978)"
2,UPDATE,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2000, numDeletionVectorsUpdated -> 0, scanTimeMs -> 692, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1859, rewriteTimeMs -> 1306)"
3,DELETE,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 954, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 535, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 417)"
4,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2100, numTargetBytesRemoved -> 1978, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 2059, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 882, numTargetRowsUpdated -> 5, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1139)"
5,UPDATE,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6137, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 831, numDeletionVectorsUpdated -> 0, scanTimeMs -> 8, numAddedFiles -> 1, numUpdatedRows -> 29, numAddedBytes -> 2191, rewriteTimeMs -> 823)"
6,RESTORE,"Map(numRestoredFiles -> 3, removedFilesSize -> 2191, numRemovedFiles -> 1, restoredFilesSize -> 6137, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 3, tableSizeAfterRestore -> 6137)"
7,OPTIMIZE,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6137, p25FileSize -> 2255, numDeletionVectorsRemoved -> 1, minFileSize -> 2255, numAddedFiles -> 1, maxFileSize -> 2255, p75FileSize -> 2255, p50FileSize -> 2255, numAddedBytes -> 2255)"
8,VACUUM START,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)"
9,VACUUM END,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)"
